# The parameter $\lambda$ in the stochastic SKT system: numerical experiments

This notebook reproduces the numerical illustration (Section 7, Figures 1 and 2) of

> F. Huber, *Existence of martingale solutions to a stochastic Shigesada–Kawasaki–Teramoto cross-diffusion system* (revised version).

It addresses two questions:

1. **How much does the solution depend on $\lambda$?** The paper's equation contains the $\lambda$-modified Itô–Stratonovich correction $\frac{\lambda}{N}\mathcal T(u)$. Here $\lambda=0$ is Itô and $\lambda=\tfrac12$ is Stratonovich, and the existence theorem needs $\lambda>\tfrac12$.
2. **Which $\lambda$ does the underlying particle system select at order $N^{-1}$?** This is the moderately interacting particle system of Chen, Daus, Holzinger and Jüngel.

**Summary of the findings** (setting below):

* $\lambda$ shifts the mean by exactly the deterministic effect of the drift $\frac{\lambda}{N}\mathcal T$.
* The variance does not depend on $\lambda$.
* Pathwise, the effect of $\lambda$ is below $1.3\%$ of the fluctuations for $N\ge 1000$.
* The particle system matches the **Itô** equation when the noise scale equals the interaction range ($B^\eta(0)=1/h$).
* It matches $\lambda\approx 0.39\pm0.10$ when the noise is correlated over **twice** the interaction range. In this sense $\lambda$ acts as a renormalization constant.

**Runtime.** With `FULL = False` (default), everything runs in about 10–15 minutes on a standard Colab CPU. The error bars are then about 2–3 times larger than in the paper, which is enough to see the $\lambda$-dependence and the Itô match but not to resolve $\lambda^*$ for the half-range case. With `FULL = True`, you get the sample sizes of the paper; this takes several hours.

The code uses `numpy`, `numba` and `matplotlib`, which are all preinstalled on Colab.

In [2]:
# !pip install numba   # uncomment if numba is not available
import numpy as np, numba as nb, time, json
import matplotlib.pyplot as plt
print('numba', nb.__version__, '| threads:', nb.config.NUMBA_NUM_THREADS)

numba 0.68.1 | threads: 32


## 1. Setting

* **Domain and species.** The domain is the torus $[0,1)$ with $n=2$ species.
* **Equation.** We solve $\partial_t u_i=\partial_{xx}\big(u_i\widetilde A_i(u)\big)$ with $\widetilde A_i(u)=a_{i0}+\sum_j a_{ij}u_j$.
* **Coefficients.** $a_{10}=a_{20}=\tfrac12$, $a_{11}=a_{22}=1$, $a_{12}=\tfrac12$, $a_{21}=\tfrac14$. These satisfy detailed balance with $\pi=(1,2)$, i.e. $\pi_1a_{12}=\pi_2a_{21}$.
* **Initial data and time.** $u_{1,2}(0)=1\pm 0.3\cos 2\pi x$, final time $T=0.01$.
* **Observable.** $m=\langle u_1(T),\cos 2\pi x\rangle$.

Set `FULL = True` to use the sample sizes of the paper.

In [3]:
FULL = False          # True: sample sizes of the paper (several hours)

A0  = np.array([0.5, 0.5])
A   = np.array([[1.0, 0.5],
                [0.25, 1.0]])
PI  = np.array([1.0, 2.0])
AMP = 0.3
T_END = 0.01
G   = 16              # SPDE cells, h = 1/16
h   = 1.0 / G
assert abs(PI[0]*A[0,1] - PI[1]*A[1,0]) < 1e-14   # detailed balance

def cos_weights(G):
    # weights w_c with <u, cos 2 pi x> = sum_c u_c w_c for cell averages u_c
    e = np.arange(G + 1) / G
    return (np.sin(2*np.pi*e[1:]) - np.sin(2*np.pi*e[:-1])) / (2*np.pi)

@nb.njit
def sample_init(N, amp):
    # N i.i.d. particles per species from u(0) (rejection sampling); returns X[2,N]
    X = np.empty((2, N))
    for i in range(2):
        sgn = 1.0 if i == 0 else -1.0
        k = 0
        while k < N:
            x = np.random.random()
            if np.random.random() * (1 + amp) < 1 + sgn * amp * np.cos(2*np.pi*x):
                X[i, k] = x; k += 1
    return X

@nb.njit
def bin_particles(X, G):
    N = X.shape[1]; hh = 1.0 / G
    u = np.zeros((2, G))
    for i in range(2):
        for k in range(N):
            u[i, int(X[i, k] * G) % G] += 1.0 / (N * hh)
    return u

## 2. The SPDE and its discretization

We discretize the paper's equation
$$
\mathrm du_i=\partial_{xx}\big(u_i\widetilde A_i(u)\big)\,\mathrm dt
+N^{-1/2}\,\partial_x\Big(\sigma_{ii}(u)\sum_k e_k\,\mathrm dW_k\Big)
+\frac{\lambda}{N}\,\mathcal T(u)_i\,\mathrm dt,\qquad \sigma_{ii}(u)=\sqrt{u_i\widetilde A_i(u)},
$$
by finite volumes (cell width $h$) and the Euler–Maruyama scheme in time.

* **Noise.** The noise is white on the dual cells, $e_k=\sqrt{2/h}\,\mathbf 1_{\text{dual cell }k}$, so that $F_1=\sum_k e_k^2=2/h$ and $F_2=0$. Its quadratic variation $\frac2N\int u_i\widetilde A_i|\varphi'|^2$ is the same as that of the particle fluctuation martingale.
* **Correction term.** It is implemented in the closed form (eq. (5.34) of the paper)
$$
\mathcal T(u)_i=\partial_x\Big[F_1\Big(\tfrac14\vartheta_i\,\partial_x\log u_i+\tfrac{\vartheta_i}{4\widetilde A_i}\,\partial_x\widetilde A_i\Big)\Big],\qquad \vartheta_i=\widetilde A_i+a_{ii}u_i .
$$
* **Common noise.** All values of $\lambda$ in a run are driven by the **same** noise increments and the same initial data.

In [ ]:
@nb.njit
def spde_step(u, lam, N, hh, dt, xi, a0, a, F1):
    # one Euler-Maruyama step of the finite-volume scheme; returns #negative cells
    Gc = u.shape[1]
    du = np.zeros_like(u)
    for i in range(2):
        At = np.empty(Gc); up = np.empty(Gc)
        for c in range(Gc):
            up[c] = max(u[i, c], 1e-10)
            At[c] = a0[i] + a[i, 0]*max(u[0, c], 0.0) + a[i, 1]*max(u[1, c], 0.0)
        q = up * At
        for c in range(Gc):
            du[i, c] += dt * (q[(c+1) % Gc] - 2*q[c] + q[(c-1) % Gc]) / hh**2
        flux = np.empty(Gc); noise = np.empty(Gc)
        for c in range(Gc):                       # edge c+1/2
            cp = (c + 1) % Gc
            Ae = 0.5*(At[c] + At[cp]); ue = 0.5*(up[c] + up[cp])
            vt = Ae + a[i, i]*ue
            flux[c] = F1 * (0.25*vt*(np.log(up[cp]) - np.log(up[c]))/hh + vt/(4*Ae)*(At[cp] - At[c])/hh)
            noise[c] = np.sqrt(2*ue*Ae) * xi[i, c] * np.sqrt(dt/(N*hh))
        for c in range(Gc):
            cm = (c - 1) % Gc
            du[i, c] += dt*lam/N*(flux[c] - flux[cm])/hh + (noise[c] - noise[cm])/hh
    u += du
    nneg = 0
    for i in range(2):
        for c in range(Gc):
            if u[i, c] < 0: nneg += 1
    return nneg

@nb.njit(parallel=True)
def spde_run(M, N, G, dt, nsteps, lams, w, a0, a, amp):
    # M independent runs; all lambdas share initial data and noise.
    nl = lams.shape[0]; hh = 1.0/G; F1 = 2.0/hh
    m = np.zeros((M, nl)); m0 = np.zeros(M)
    prof = np.zeros((M, nl, G)); neg = np.zeros((M, nl), dtype=np.int64)
    for r in nb.prange(M):
        ub = bin_particles(sample_init(N, amp), G)
        m0[r] = np.sum(ub[0]*w)
        us = np.empty((nl, 2, G))
        for l in range(nl): us[l] = ub
        for s in range(nsteps):
            xi = np.random.standard_normal((2, G))
            for l in range(nl):
                neg[r, l] += spde_step(us[l], lams[l], N, hh, dt, xi, a0, a, F1)
        for l in range(nl):
            prof[r, l] = us[l, 0]
            m[r, l] = np.sum(us[l, 0]*w)
    return m, m0, prof, neg

def spde_deterministic(G, dt, nsteps, lam=0.0, N=1e300):
    # N = infinity limit (lam=0), or the deterministic effect of the drift (lam/N) T (no noise)
    hh = 1.0/G; e = np.arange(G+1)/G
    s = (np.sin(2*np.pi*e[1:]) - np.sin(2*np.pi*e[:-1]))/(2*np.pi*hh)
    u = np.stack([1 + AMP*s, 1 - AMP*s]); xi = np.zeros((2, G))
    for _ in range(nsteps): spde_step(u, lam, float(N), hh, dt, xi, A0, A, 2.0/hh)
    return u

## 3. The particle system

The particle system (with linear interaction) is
$$
\mathrm dX^k_i=\sqrt{2\Theta_i(X^k_i)}\,\mathrm dW^k_i,\qquad
\Theta_i=a_{i0}+\sum_j a_{ij}\,B^\eta*\mu^N_j,\qquad \mu^N_j=\tfrac1N\textstyle\sum_l\delta_{X^l_j}.
$$
* **Mollifier.** $B^\eta$ is a periodic Gaussian. The self-interaction term $l=k$ is included.
* **Mean-field limit.** $\partial_tu_i=\partial_{xx}(u_i\Theta_i(u))$.
* **Time stepping.** Euler–Maruyama with $\sqrt{2\Theta\,\mathrm dt}\ll\eta$.
* **Convolution.** $B^\eta*\mu^N$ is evaluated by cloud-in-cell deposition on a fine grid with $G_f$ points.
* **Scales.** We use $\eta=h/\sqrt{2\pi}$, so that $B^\eta(0)=1/h$ (matched scales), and $\eta=h/(2\sqrt{2\pi})$, so that $B^\eta(0)=2/h$ (noise correlated over twice the interaction range).

In [ ]:
def kernel(eta, Gf):
    hf = 1.0/Gf; K = int(np.ceil(5*eta/hf))
    x = np.arange(-K, K+1)*hf
    B = sum(np.exp(-(x+img)**2/(2*eta**2)) for img in (-1, 0, 1)) / np.sqrt(2*np.pi*eta**2)
    return B*hf, K

@nb.njit
def theta_at(X, Bw, K, Gf, a0, a):
    N = X.shape[1]; hf = 1.0/Gf
    dens = np.zeros((2, Gf))
    for j in range(2):
        for k in range(N):
            y = X[j, k]*Gf - 0.5; c = int(np.floor(y)); f = y - c
            dens[j, c % Gf] += (1-f)/(N*hf); dens[j, (c+1) % Gf] += f/(N*hf)
    conv = np.zeros((2, Gf))
    for j in range(2):
        for c in range(Gf):
            s = 0.0
            for mm in range(-K, K+1): s += Bw[mm+K]*dens[j, (c-mm) % Gf]
            conv[j, c] = s
    th = np.empty((2, N))
    for i in range(2):
        for k in range(N):
            y = X[i, k]*Gf - 0.5; c = int(np.floor(y)); f = y - c
            v = a0[i]
            for j in range(2): v += a[i, j]*((1-f)*conv[j, c % Gf] + f*conv[j, (c+1) % Gf])
            th[i, k] = v
    return th

@nb.njit(parallel=True)
def particle_run(M, N, dt, nsteps, Bw, K, Gf, a0, a, amp):
    m = np.zeros(M); m0 = np.zeros(M)
    for r in nb.prange(M):
        X = sample_init(N, amp)
        m0[r] = np.mean(np.cos(2*np.pi*X[0]))
        for s in range(nsteps):
            th = theta_at(X, Bw, K, Gf, a0, a)
            Z = np.random.standard_normal((2, N))
            for i in range(2):
                for k in range(N):
                    X[i, k] = (X[i, k] + np.sqrt(2*th[i, k]*dt)*Z[i, k]) % 1.0
        m[r] = np.mean(np.cos(2*np.pi*X[0]))
    return m, m0

def mean_field(eta, Gf, dt, nsteps):
    # mollified mean-field PDE (FFT convolution, finite differences)
    hf = 1.0/Gf; e = np.arange(Gf+1)*hf
    s = (np.sin(2*np.pi*e[1:]) - np.sin(2*np.pi*e[:-1]))/(2*np.pi*hf)
    u = np.stack([1 + AMP*s, 1 - AMP*s])
    Bh = np.exp(-2*(np.pi*np.fft.rfftfreq(Gf, d=hf)*eta)**2)
    for _ in range(nsteps):
        Bu = np.fft.irfft(np.fft.rfft(u, axis=1)*Bh, n=Gf, axis=1)
        q = u*(A0[:, None] + A @ Bu)
        u = u + dt*(np.roll(q, -1, 1) - 2*q + np.roll(q, 1, 1))/hf**2
    w = (np.sin(2*np.pi*e[1:]) - np.sin(2*np.pi*e[:-1]))/(2*np.pi)
    return u[0] @ w

## 4. Statistics

The bias is $\mathbb E m-m_{\rm det}$, where $m_{\rm det}$ is the $N=\infty$ limit of the respective model, computed with the same discretization. We use the initial value $m(0)$, whose mean is known exactly, as a control variate. This removes the variance coming from the random initial data.

In [ ]:
def control_variate(m, m0, m0_exact):
    beta = np.cov(m, m0)[0, 1] / np.var(m0)
    return m - beta*(m0 - m0_exact)

def bias_stats(m, m0, m0_exact, m_det, N):
    mc = control_variate(m, m0, m0_exact)
    return N*(mc.mean() - m_det), N*mc.std()/np.sqrt(len(m))

## 5. Experiment 1: dependence on $\lambda$ (SPDE only)

We use $\lambda\in\{0,\tfrac14,\tfrac12,1\}$ and $N\in\{250,\dots,4000\}$, with common noise across $\lambda$. For each $N$ we compute:

* the bias of the mean;
* the shift $N(\mathbb E m_\lambda-\mathbb E m_0)$, compared with the deterministic effect of $\frac\lambda N\mathcal T$ alone;
* the variance $N\,\mathrm{Var}(m)$;
* the pathwise effect $\|u^\lambda_1-u^0_1\|_{L^2}$ relative to the fluctuations $\|u^0_1-\mathbb E u^0_1\|_{L^2}$.

The paper used $4\cdot10^5$ runs for $N=250,500$ and $10^5$ for $N\ge1000$.

In [ ]:
w = cos_weights(G)
dt_spde = 1e-4; ns = int(round(T_END/dt_spde))
lams = np.array([0.0, 0.25, 0.5, 1.0])
e = np.arange(G+1)/G; s = (np.sin(2*np.pi*e[1:]) - np.sin(2*np.pi*e[:-1]))/(2*np.pi*h)
m0_exact_spde = AMP*s @ w                                  # E m(0) for binned data
m_det_spde = spde_deterministic(G, dt_spde, ns)[0] @ w

M_spde = {250: 400000, 500: 400000, 1000: 100000, 2000: 100000, 4000: 100000} if FULL else \
         {250: 40000, 500: 40000, 1000: 20000, 2000: 20000, 4000: 20000}
spde_res = {}
for N, M in M_spde.items():
    t0 = time.time(); ms, m0s, negs = [], [], 0
    sum_d2 = np.zeros(len(lams)); prof_store = []
    for b in range(max(1, M//20000)):
        Mb = min(20000, M)
        m, m0, prof, neg = spde_run(Mb, N, G, dt_spde, ns, lams, w, A0, A, AMP)
        ms.append(m); m0s.append(m0); negs = negs + neg.sum(0)
        sum_d2 += h*((prof - prof[:, :1, :])**2).sum(-1).sum(0)
        prof_store.append(prof[:, 0, :])
    m = np.concatenate(ms); m0 = np.concatenate(m0s); p0 = np.concatenate(prof_store)
    path = np.sqrt(sum_d2/len(m))
    fluct = np.sqrt((h*((p0 - p0.mean(0))**2).sum(-1)).mean())
    pred = [N*(spde_deterministic(G, dt_spde, ns, lam, N)[0] @ w - m_det_spde) for lam in lams]
    spde_res[N] = dict(
        bias=[bias_stats(m[:, l], m0, m0_exact_spde, m_det_spde, N) for l in range(len(lams))],
        shift=N*(m - m[:, :1]).mean(0), pred=np.array(pred), var=N*m.var(0),
        path_ratio=path/fluct, neg=negs)
    print(f"N={N:5d}  runs={len(m):6d}  ({time.time()-t0:5.1f}s)")
    print("   N*bias      :", " ".join(f"{b:+.3f}±{e:.3f}" for b, e in spde_res[N]['bias']))
    print("   N*shift     :", np.round(spde_res[N]['shift'], 4), "  deterministic:", np.round(spde_res[N]['pred'], 4))
    print("   N*Var       :", np.round(spde_res[N]['var'], 4))
    print("   path/fluct  :", np.round(spde_res[N]['path_ratio'], 5), "  negative cells:", negs)

## 6. Experiment 2: particles against the SPDE

We compare the $O(N^{-1})$ bias of the particle system with that of the SPDE as a function of $\lambda$. The value of $\lambda$ that matches is
$$
\lambda^*=\frac{b_{\rm part}-b_{\rm SPDE}(0)}{b_{\rm SPDE}(1)-b_{\rm SPDE}(0)}\quad(\text{the SPDE bias is linear in }\lambda).
$$

Paper values ($N\cdot$bias):

| model | $N\cdot$bias |
|---|---|
| SPDE, $h=1/16$, $N=250$, $\lambda=0,\frac14,\frac12,1$ | $-0.186,\ -0.353,\ -0.518,\ -0.842\ (\pm0.014)$ |
| particles, $B^\eta(0)=1/h$, $N=250$ | $-0.185\pm0.036$ |
| particles, $B^\eta(0)=1/h$, $N=500$ | $-0.271\pm0.062$ |
| particles, $B^\eta(0)=2/h$, $N=250$ | $-0.443\pm0.063$, which gives $\lambda^*=0.39\pm0.10$ |

In [ ]:
part_cases = [  # (label, eta, Gf, dt, N, M_full, M_quick)
    ('matched,  B(0)=1/h, N=250', h/np.sqrt(2*np.pi),     256, 2e-5, 250, 60000, 4000),
    ('matched,  B(0)=1/h, N=500', h/np.sqrt(2*np.pi),     256, 2e-5, 500, 40000, 2000),
    ('half eta, B(0)=2/h, N=250', h/(2*np.sqrt(2*np.pi)), 512, 5e-6, 250, 20000, 2000),
]
part_res = {}
for lab, eta, Gf, dt, N, Mf, Mq in part_cases:
    M = Mf if FULL else Mq
    Bw, K = kernel(eta, Gf); nsteps = int(round(T_END/dt))
    m_mf = mean_field(eta, 512, 2.5e-7 if eta > 0.02 else 6e-8, int(round(T_END/(2.5e-7 if eta > 0.02 else 6e-8))))
    t0 = time.time(); ms, m0s = [], []
    for b in range(max(1, M//1000)):
        m, m0 = particle_run(min(1000, M), N, dt, nsteps, Bw, K, Gf, A0, A, AMP); ms.append(m); m0s.append(m0)
    m = np.concatenate(ms); m0 = np.concatenate(m0s)
    b, se = bias_stats(m, m0, AMP/2, m_mf, N)          # <u(0), cos 2 pi x> = AMP/2
    sp = spde_res[N]['bias']
    lam_star = (b - sp[0][0])/(sp[3][0] - sp[0][0])
    part_res[lab] = dict(N=N, bias=b, se=se, var=N*m.var(), lam_star=lam_star,
                         lam_star_se=se/abs(sp[3][0] - sp[0][0]))
    print(f"{lab}: runs={len(m)} ({time.time()-t0:.0f}s)  N*bias = {b:+.3f} ± {se:.3f}   N*Var = {N*m.var():.3f}"
          f"   matching lambda* = {lam_star:.2f} ± {part_res[lab]['lam_star_se']:.2f}")

## 7. Figures (Figures 1 and 2 of the paper)

In [ ]:
plt.rcParams.update({'font.size': 9, 'mathtext.fontset': 'cm'})
fig, ax = plt.subplots(1, 2, figsize=(9, 3.4))
mk = {250: 'o', 500: 's', 1000: '^', 2000: 'v', 4000: 'D'}
for N, r in spde_res.items():
    ax[0].plot(lams, r['shift'], mk[N], mfc='none', label=f'$N={N}$')
ax[0].plot(lams, spde_res[250]['pred'], 'k-', lw=0.8, label='deterministic')
ax[0].set_xlabel(r'$\lambda$'); ax[0].set_ylabel(r'$N(\mathbb{E}m_\lambda-\mathbb{E}m_0)$'); ax[0].legend(fontsize=7)
Ns = np.array([N for N in spde_res if N >= 1000])
for l, L in [(2, 0.5), (3, 1.0)]:
    ax[1].loglog(Ns, [spde_res[N]['path_ratio'][l] for N in Ns], 'o-', label=rf'$\lambda={L:g}$')
ax[1].loglog(Ns, spde_res[Ns[0]]['path_ratio'][3]*(Ns/Ns[0])**-0.5, 'k:', label=r'$\propto N^{-1/2}$')
ax[1].set_xlabel('$N$'); ax[1].set_ylabel(r'$\|u^\lambda-u^0\|/\|u^0-\mathbb{E}u^0\|$'); ax[1].legend(fontsize=7)
plt.tight_layout(); plt.savefig('fig_lambda.png', dpi=150); plt.show()

fig, ax = plt.subplots(figsize=(5.5, 3.4))
for N, sty in [(250, 'o-k'), (500, 's--')]:
    b = np.array(spde_res[N]['bias'])
    ax.errorbar(lams, b[:, 0], yerr=2*b[:, 1], fmt=sty, capsize=3, label=f'SPDE, $h=1/16$, $N={N}$')
for (lab, r), col in zip(part_res.items(), ['C0', 'C2', 'C3']):
    ax.axhspan(r['bias'] - 2*r['se'], r['bias'] + 2*r['se'], color=col, alpha=0.15)
    ax.axhline(r['bias'], color=col, label='particles, ' + lab)
ax.set_xlabel(r'$\lambda$'); ax.set_ylabel(r'$N(\mathbb{E}m-m_{\rm det})$'); ax.legend(fontsize=7)
plt.tight_layout(); plt.savefig('fig_bias.png', dpi=150); plt.show()

In [ ]:
# save the numbers
out = {'spde': {str(N): {k: (np.asarray(v).tolist()) for k, v in r.items()} for N, r in spde_res.items()},
       'particles': part_res, 'FULL': FULL}
json.dump(out, open('results.json', 'w'), indent=1, default=float)
print('saved results.json, fig_lambda.png, fig_bias.png')

## 8. Interpretation and caveats

* **Self-interaction.** In the particle system, $\Theta_i$ contains the self-interaction $\frac{a_{ii}}{N}B^\eta(0)$. In the Itô SPDE, the corresponding term comes from the fluctuations of the cell averages, $\mathbb E[u_i\widetilde A_i(u)]\ni a_{ii}\operatorname{Var}(u_{i,c})\approx a_{ii}u_i/(Nh)$. The two agree when $B^\eta(0)=1/h$, which explains the match at $\lambda=0$.
* **Role of $\lambda$.** The correction $\frac\lambda N\mathcal T$ is of the same order $\frac1{Nh}$, since $F_1=2/h$. When the noise is smoothed over a longer range than the interaction, it compensates for the missing part of the self-interaction. So $\lambda$ acts as a renormalization constant fixed by the ratio of the two scales, and $\lambda>\tfrac12$ corresponds to noise correlated well beyond the interaction range.
* **Caveats.** The setting is one-dimensional, periodic, with one observable and short times, and the results depend on how the noise is discretized. This is an illustration, not a proof.